# Monarch · Stickman film (image → video)

Tareeqa wohi jo aap ne kaha: **pehle 12 stickman images**, phir un ko **I2V model** se animate.

**2 tap:** (1) `Runtime → Change runtime type → T4 GPU` → Save, (2) `Runtime → Run all`.
Clips Google Drive me `Monarch-T2V/stick-clips/` me jati hain (resume-safe). Aap koi command nahi chalayenge.

Model: **Wan 2.2 TI2V-5B**. Agar koi scene fail ho to dobara Run all — jo ban chuki hai skip ho jayegi.


In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU chuno, phir Run all"
p = torch.cuda.get_device_properties(0)
print("VRAM:", round(p.total_memory / 1e9, 1), "GB | bf16 support:", torch.cuda.is_bf16_supported())


In [ ]:
%pip -q install "git+https://github.com/huggingface/diffusers" "transformers>=4.49" accelerate safetensors sentencepiece protobuf ftfy imageio imageio-ffmpeg
import diffusers, transformers
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__)


In [ ]:
import pathlib
OUT = None
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT = pathlib.Path('/content/drive/MyDrive/Monarch-T2V/stick-clips')
    OUT.mkdir(parents=True, exist_ok=True)
    print("Drive folder ready:", OUT)
except Exception as exc:
    print("Drive mount nahi hua:", type(exc).__name__, exc, "— aap approve karke yeh cell dobara chala dein")


In [ ]:
import pathlib, urllib.request, json

KF = pathlib.Path('/content/keyframes'); KF.mkdir(exist_ok=True)
RAW = "https://raw.githubusercontent.com/adil-chandio/Monarch-Studio/arena/3d8cfad7-monarch-studio/studio/films/stick-i2v/keyframes"
for i in range(1, 13):
    sid = f"k{i:02d}"
    f = KF / f"{sid}.png"
    if not f.exists() or f.stat().st_size < 10000:
        urllib.request.urlretrieve(f"{RAW}/{sid}.png", f)
print(len(list(KF.glob('*.png'))), "keyframes ready in", KF)

if OUT is None:
    OUT = pathlib.Path('/content/stick-clips'); OUT.mkdir(exist_ok=True)
print("clips jayengi:", OUT)

SCENES = [
 {
  "id": "k01",
  "seed": 201,
  "prompt": "the chalk stick figure walks forward toward the camera through falling rain, a natural walking cycle with swinging arms and legs, rain streaks sliding down past him, tiny chalk splashes at his feet. hand-drawn white chalk stickman animation on a near-black chalkboard, exactly the same chalk character as the first frame with a small gold cap, chalky organic line quality, gentle chalk boil, drifting dust motes, cinematic lighting, no text"
 },
 {
  "id": "k02",
  "seed": 202,
  "prompt": "the chalk stick figure stands at a rainy window and slowly lowers his head, his shoulders sink, rain runs down the glass behind him, slow heavy breathing. hand-drawn white chalk stickman animation on a near-black chalkboard, exactly the same chalk character as the first frame with a small gold cap, chalky organic line quality, gentle chalk boil, drifting dust motes, cinematic lighting, no text"
 },
 {
  "id": "k03",
  "seed": 203,
  "prompt": "the chalk stick figure leans across a cafe table toward a second figure and gestures with one hand while he speaks, the other figure nods slightly, steam curls up from a cup. hand-drawn white chalk stickman animation on a near-black chalkboard, exactly the same chalk character as the first frame with a small gold cap, chalky organic line quality, gentle chalk boil, drifting dust motes, cinematic lighting, no text"
 },
 {
  "id": "k04",
  "seed": 204,
  "prompt": "close on the chalk stick figure as he speaks calmly, his mouth forms words, his eyebrows move, one hand rises and falls with the rhythm of his speech. hand-drawn white chalk stickman animation on a near-black chalkboard, exactly the same chalk character as the first frame with a small gold cap, chalky organic line quality, gentle chalk boil, drifting dust motes, cinematic lighting, no text"
 },
 {
  "id": "k05",
  "seed": 205,
  "prompt": "extreme close on the chalk stick figure's face, his eyes blink slowly once, his jaw tightens with determination, chalk dust drifts across the frame. hand-drawn white chalk stickman animation on a near-black chalkboard, exactly the same chalk character as the first frame with a small gold cap, chalky organic line quality, gentle chalk boil, drifting dust motes, cinematic lighting, no text"
 },
 {
  "id": "k06",
  "seed": 206,
  "prompt": "the chalk stick figure pulls his shoulders back and stands tall, his chest opens as he takes one slow breath, golden rays pulse gently behind him. hand-drawn white chalk stickman animation on a near-black chalkboard, exactly the same chalk character as the first frame with a small gold cap, chalky organic line quality, gentle chalk boil, drifting dust motes, cinematic lighting, no text"
 },
 {
  "id": "k07",
  "seed": 207,
  "prompt": "the chalk stick figure sits still at a long table while blurred grey figures gesture and argue around him, papers shift on the table, he does not move. hand-drawn white chalk stickman animation on a near-black chalkboard, exactly the same chalk character as the first frame with a small gold cap, chalky organic line quality, gentle chalk boil, drifting dust motes, cinematic lighting, no text"
 },
 {
  "id": "k08",
  "seed": 208,
  "prompt": "the chalk stick figures laugh around a table, the hero throws his head back in laughter while the others rock with him, warm light pulses. hand-drawn white chalk stickman animation on a near-black chalkboard, exactly the same chalk character as the first frame with a small gold cap, chalky organic line quality, gentle chalk boil, drifting dust motes, cinematic lighting, no text"
 },
 {
  "id": "k09",
  "seed": 209,
  "prompt": "the chalk stick figure practices speaking alone in a dim room at dawn, he gestures as he talks, his dim reflection on the right side mirrors his movement, dust drifts. hand-drawn white chalk stickman animation on a near-black chalkboard, exactly the same chalk character as the first frame with a small gold cap, chalky organic line quality, gentle chalk boil, drifting dust motes, cinematic lighting, no text"
 },
 {
  "id": "k10",
  "seed": 210,
  "prompt": "the chalk stick figure stands steady in front of a rising golden sun, the sun pulses and its rays rotate slowly behind him, dust drifts past. hand-drawn white chalk stickman animation on a near-black chalkboard, exactly the same chalk character as the first frame with a small gold cap, chalky organic line quality, gentle chalk boil, drifting dust motes, cinematic lighting, no text"
 },
 {
  "id": "k11",
  "seed": 211,
  "prompt": "the chalk stick figure walks steadily forward while a crowd of grey figures moves the other way and blurs past him, his stride is confident. hand-drawn white chalk stickman animation on a near-black chalkboard, exactly the same chalk character as the first frame with a small gold cap, chalky organic line quality, gentle chalk boil, drifting dust motes, cinematic lighting, no text"
 },
 {
  "id": "k12",
  "seed": 212,
  "prompt": "a vintage chalk microphone in a dark studio, dust drifts slowly through a beam of light, a small green tick appears and glows. hand-drawn white chalk stickman animation on a near-black chalkboard, exactly the same chalk character as the first frame with a small gold cap, chalky organic line quality, gentle chalk boil, drifting dust motes, cinematic lighting, no text"
 }
]
NEG = "photorealistic, 3d render, cgi, colored skin, extra limbs, melting lines, morphing, blurry, watermark, text, frozen static frame"
print(len(SCENES), "scenes")


In [ ]:
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import load_image, export_to_video

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
MODEL = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()
pipe.set_progress_bar_config(disable=True)
print("Wan 2.2 TI2V-5B ready | dtype:", DTYPE)


In [ ]:
import time

FRAMES = 73          # 73 = 4k+1 -> ~3.0s @24fps
STEPS = 22
H, W = 832, 480

def render(scene):
    img = load_image(str(KF / f'{scene["id"]}.png')).resize((W, H))
    with torch.no_grad():
        return pipe(
            prompt=scene["prompt"],
            negative_prompt=NEG,
            image=img,
            height=H, width=W,
            num_frames=FRAMES,
            num_inference_steps=STEPS,
            guidance_scale=5.0,
            generator=torch.Generator("cpu").manual_seed(int(scene["seed"])),
        ).frames[0]

results = []
todo = [s for s in SCENES if not ((OUT / f'{s["id"]}.mp4').exists() and (OUT / f'{s["id"]}.mp4').stat().st_size > 20000)]
print(f"{len(todo)} scenes baqi")
for n, scene in enumerate(todo, 1):
    t0 = time.time()
    print(f'[{n}/{len(todo)}] {scene["id"]} animate ho rahi hai...', flush=True)
    try:
        frames = render(scene)
        tmp = pathlib.Path('/content') / f'{scene["id"]}.mp4'
        export_to_video(frames, str(tmp), fps=24)
        (OUT / f'{scene["id"]}.mp4').write_bytes(tmp.read_bytes())
        dt = round(time.time() - t0)
        print(f'{scene["id"]} OK in {dt//60}m {dt%60}s', flush=True)
        results.append({"scene": scene["id"], "state": "ok", "seconds": dt})
    except Exception as exc:
        print(f'{scene["id"]} FAIL {type(exc).__name__}: {str(exc)[:220]}', flush=True)
        results.append({"scene": scene["id"], "state": "fail", "error": f"{type(exc).__name__}: {str(exc)[:220]}"})
    (OUT / "receipt.json").write_text(json.dumps(results, indent=2))


In [ ]:
done = sorted(p.stem for p in OUT.glob('k*.mp4'))
print(f"{len(done)} / {len(SCENES)} clips ready:", ", ".join(done))
missing = [s["id"] for s in SCENES if s["id"] not in done]
print("baqi:", missing if missing else "kuch nahi")
if missing:
    print("\nAgar koi scene fail hui ho: dobara Run all karein — bani hui clips skip ho jayengi.")
print("\nAgent ko bolen: 'stick clips ready' — final film main banaunga.")
